# Train the vertebra corner refiner on Colab

This notebook copies the existing private `MyDrive/SOS Colab` package to fast Colab storage, validates the full COCO dataset, gates training with an optional fixed 32-crop overfit test, trains HRNet-W18, and atomically synchronizes essential artifacts to Drive after every epoch. It does not create another dataset copy in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil

DRIVE_PACKAGE = Path('/content/drive/MyDrive/SOS Colab')
PROJECT_ROOT = Path('/content/Spine-Opportunistic-Screening')
LOCAL_RUN_PARENT = Path('/content/refiner_runs')
DRIVE_RUN_PARENT = Path('/content/drive/MyDrive/spine_refiner_runs')
DRIVE_CENTERNET_CHECKPOINT = Path('/content/drive/MyDrive/spine_centernet_weights/best_center_f1.pt')
EXPERIMENT = 'hrnet_w18_dsnt_full_coco'
RUN_OVERFIT_TEST = True
OVERFIT_GATE_PASSED = False
RESUME = True

if not DRIVE_PACKAGE.is_dir():
    raise FileNotFoundError(DRIVE_PACKAGE)
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
shutil.copytree(DRIVE_PACKAGE, PROJECT_ROOT)
%cd /content/Spine-Opportunistic-Screening
print('project:', PROJECT_ROOT)

In [ ]:
!pip install -q -r requirement.txt
import platform, cv2, numpy as np, torch, timm
print('python', platform.python_version())
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('timm', timm.__version__, 'opencv', cv2.__version__, 'numpy', np.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before training.')

## Dataset and model contract checks

The first command verifies full train counts and file resolution. The next two verify validation/test counts. The model-forward check covers ROI round trips, isotropic scale, black padding, corner containment/order preservation, target encode/decode, output shape, finite loss, and nonzero gradients.

In [ ]:
!python -m src.workflows.check_corner_refiner_dataset --dataset-root dataset --split train --limit 8 --verify-full-counts --verify-all-files --model-forward
!python -m src.workflows.check_corner_refiner_dataset --dataset-root dataset --split val --limit 4 --verify-full-counts --verify-all-files
!python -m src.workflows.check_corner_refiner_dataset --dataset-root dataset --split test --limit 4 --verify-full-counts --verify-all-files

## Compare deterministic and training-jitter crops

In [ ]:
import matplotlib.pyplot as plt
from src.data.corner_refiner_dataset import CornerRefinerCocoDataset, IMAGENET_MEAN, IMAGENET_STD

torch.manual_seed(20260627)
fixed = CornerRefinerCocoDataset('dataset', 'train', augment=False, limit=8)
jittered = CornerRefinerCocoDataset('dataset', 'train', augment=True, limit=8)
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for column in range(8):
    for row, sample in enumerate((fixed[column], jittered[column])):
        image = sample['input'].numpy() * IMAGENET_STD + IMAGENET_MEAN
        image = np.clip(image.transpose(1, 2, 0), 0, 1)
        points = sample['target_points_crop'].numpy()
        axes[row, column].imshow(image)
        axes[row, column].plot(points[[0,1,3,2,0],0], points[[0,1,3,2,0],1], 'y-', lw=1)
        axes[row, column].scatter(points[:,0], points[:,1], c=['r','lime','cyan','magenta'], s=12)
        axes[row, column].set_title(('fixed' if row == 0 else 'jitter') + f' V{sample["vertebra_rank"]}')
        axes[row, column].axis('off')
plt.tight_layout()

## Mandatory 32-crop overfit gate

Leave `RUN_OVERFIT_TEST=True` for the first run. The command exits with an error unless mean crop error is below 2 px and NME is below 0.01. A failure must be investigated before starting the full run.

In [ ]:
import subprocess, sys
if RUN_OVERFIT_TEST:
    subprocess.run([
        sys.executable, '-u', '-m', 'src.train_corner_refiner',
        '--dataset-root', 'dataset', '--output-dir', '/content/refiner_overfit', '--experiment-name', 'overfit_32',
        '--overfit-samples', '32', '--epochs', '150', '--batch-size', '16',
        '--num-workers', '2', '--early-stop-patience', '0'
    ], check=True)
    OVERFIT_GATE_PASSED = True
else:
    print('WARNING: overfit gate skipped intentionally; do not start a first full run without passing it.')

## Restore and train

Training writes to `/content/refiner_runs`. The CLI atomically copies `last.pt`, both best checkpoints, the CSV log, configuration, and validation metrics to Drive after every epoch.

In [ ]:
local_run = LOCAL_RUN_PARENT / EXPERIMENT
drive_run = DRIVE_RUN_PARENT / EXPERIMENT
if RUN_OVERFIT_TEST and not OVERFIT_GATE_PASSED:
    raise RuntimeError('The 32-crop overfit gate has not passed; full training is blocked.')
if RESUME and drive_run.is_dir():
    local_run.mkdir(parents=True, exist_ok=True)
    shutil.copytree(drive_run, local_run, dirs_exist_ok=True)
    print('restored:', drive_run)
resume_checkpoint = local_run / 'last.pt'
command = [
    sys.executable, '-u', '-m', 'src.train_corner_refiner',
    '--config', 'configs/config.yaml',
    '--dataset-root', 'dataset',
    '--output-dir', str(LOCAL_RUN_PARENT), '--experiment-name', EXPERIMENT,
    '--backup-dir', str(DRIVE_RUN_PARENT),
]
if RESUME and resume_checkpoint.is_file():
    command += ['--resume-checkpoint', str(resume_checkpoint)]
print(' '.join(command))
subprocess.run(command, check=True)

## Validate the frozen best-NME checkpoint and export inference weights

This evaluates validation only. Do not run test evaluation until the architecture, crop policy, and checkpoint choice are frozen.

In [ ]:
best_checkpoint = local_run / 'best_nme.pt'
subprocess.run([
    sys.executable, '-u', '-m', 'src.evaluate_corner_refiner',
    '--checkpoint', str(best_checkpoint), '--dataset-root', 'dataset', '--split', 'val',
    '--output-dir', str(local_run), '--export-inference'
], check=True)
for name in ('val_metrics.json', 'inference_refiner.pt'):
    source = local_run / name
    destination = drive_run / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + '.tmp')
    shutil.copy2(source, temporary)
    temporary.replace(destination)
print('validation/export synced to', drive_run)

## Two-stage matched validation acceptance

Before running this cell, place the current CenterNet checkpoint at `MyDrive/spine_centernet_weights/best_center_f1.pt`. Baseline and refined corners are compared on exactly the same matched Mendeley/MICCAI detections. The cell fails unless aggregate NME improves by at least 20% and neither source is materially degraded.

In [ ]:
if not DRIVE_CENTERNET_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f'Upload the current CenterNet checkpoint to {DRIVE_CENTERNET_CHECKPOINT}'
    )
local_centernet_checkpoint = Path('/content/best_center_f1.pt')
shutil.copy2(DRIVE_CENTERNET_CHECKPOINT, local_centernet_checkpoint)
two_stage_dir = local_run / 'two_stage_validation'
subprocess.run([
    sys.executable, '-u', '-m', 'src.evaluate_two_stage_refiner',
    '--centernet-checkpoint', str(local_centernet_checkpoint),
    '--refiner-checkpoint', str(local_run / 'inference_refiner.pt'),
    '--dataset-root', 'dataset', '--split', 'val',
    '--output-dir', str(two_stage_dir), '--enforce-acceptance'
], check=True)
drive_two_stage_dir = drive_run / 'two_stage_validation'
for name in ('two_stage_metrics.json', 'matched_instances.csv'):
    source = two_stage_dir / name
    destination = drive_two_stage_dir / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + '.tmp')
    shutil.copy2(source, temporary)
    temporary.replace(destination)
print('two-stage acceptance artifacts synced to', drive_two_stage_dir)